In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import timedelta
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set(style="whitegrid")
if not os.path.exists('plots'): os.makedirs('plots')

In [ ]:
df = pd.read_csv('dataset.csv')
df['time_dt'] = pd.to_datetime(df['transaction_time'], format='mixed')
df['new_day'] = (df['time_dt'].diff().dt.total_seconds() < 0).astype(int)
df['day_offset'] = df['new_day'].cumsum()
start_date = pd.Timestamp('2025-01-01')
df['transaction_date'] = df['day_offset'].apply(lambda x: start_date + timedelta(days=x))
df['revenue'] = df['transaction_qty'] * df['unit_price']
df['hour'] = df['time_dt'].dt.hour
df.head()

In [ ]:
plt.figure(figsize=(12, 6))
df.groupby('transaction_date')['revenue'].sum().plot()
plt.title('Daily Revenue Trend')
plt.show()

In [ ]:
daily_data = df.groupby('transaction_date')['revenue'].sum().reset_index()
daily_data.columns = ['ds', 'y']
train_size = int(len(daily_data) * 0.8)
train, test = daily_data.iloc[:train_size], daily_data.iloc[train_size:]

m = Prophet()
m.fit(train)
future = m.make_future_dataframe(periods=len(test))
forecast = m.predict(future)
m.plot(forecast)
plt.show()